# Lab 5 · Capstone — An End-to-End MS Mini-Project
**MSACL · DS301 Deep Learning · Segment 5 · Lab 5**

This is the capstone: you pull the whole course together on **real course data**. Pick **one
track**, adapt a working pipeline, and — most important — **evaluate it like a clinical lab**
(the Lecture 10 lesson): a proper train/validation split with **no leakage**, and
**sensitivity, specificity, and AUROC + a confusion matrix**, never bare accuracy.

**Three tracks (choose one by setting `TRACK`):**

- **Track A — Beat the baseline.** Start from the Lab 2 1D-CNN on the S. aureus/oxacillin
  (MRSA) spectra and **improve it** (deeper/wider conv + m/z-jitter augmentation). Compare
  improved-vs-baseline **on the same split** with proper metrics.
- **Track B — Peak QC.** Classify PeakOnly chromatographic ROIs as **real peak vs. noise**
  (the Lecture 5 object-detection segment made real), handling the class imbalance honestly.
- **Track C — New question, same spectra.** **Transfer learning** (the Lecture 8 recipe):
  take the S. aureus-trained CNN **body**, **freeze** it, and re-head it for a *different*
  organism/drug (E. coli + ceftriaxone) with a fresh trainable head (the body stays frozen) and a smaller fine-tune LR.

Each track has **2 fill-in blanks** (`YOUR CODE HERE`) — the "adapt the pipeline" choices —
plus `assert` self-checks that tell you instantly whether your choice is structurally sound.


## How this capstone works

1. **Set `TRACK`** to `'A'`, `'B'`, or `'C'` in the config cell below, then **Run All**.
   Only your track's cells do real work; the others are skipped.
2. Fill in your track's **2 blanks**. The `assert` cells check shapes, split-disjointness,
   frozen/trainable parameter counts, and that every metric lands in `[0, 1]` — they do **not**
   check accuracy magnitude, so a small run still passes.
3. Read the **honest error analysis** cell for your track, then fill the **rubric self-check**
   and the **project one-pager** at the bottom.

> **Shared, read-and-run scaffolding** (next section) gives you the data loaders, a leakage-free
> **stratified split**, and an **`honest_eval`** helper that reports sensitivity / specificity /
> AUROC + confusion matrix. You reuse these in every track — don't rewrite them.

`RUN_ALL_TRACKS` (default `False`) is a smoke switch: set it `True` to execute **all three**
tracks in one pass (used by the instructor's run test). Leave it `False` for your own work.


In [ ]:
# ============================ CONFIG — edit TRACK, then Run All ============================
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, confusion_matrix

# ---- your choice: which track are you doing? ----
TRACK = 'A'              # 'A' beat-the-baseline | 'B' peak-QC | 'C' transfer learning

# ---- instructor smoke switches (leave as-is for your own work) ----
RUN_ALL_TRACKS = False   # True = run all three tracks in one pass (run test uses this)
FORCE_CPU = False        # True = ignore GPU/MPS and run on CPU (run test uses this)
SMOKE = False            # True = tiny data subset + few epochs (run test uses this)

# ---- derived run sizes ----
EPOCHS = 2 if SMOKE else 12
MAX_N = 200 if SMOKE else None          # cap samples per dataset when smoke-testing

if FORCE_CPU:
    device = torch.device("cpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

torch.manual_seed(42)
np.random.seed(42)

def should_run(track):
    """A track's cells run if it is the chosen TRACK, or if the smoke switch is on."""
    return RUN_ALL_TRACKS or TRACK == track

print(f"TRACK={TRACK} | RUN_ALL_TRACKS={RUN_ALL_TRACKS} | SMOKE={SMOKE} | device={device} | epochs={EPOCHS}")


## Shared scaffolding — data, a leakage-free split, and honest evaluation *(read and run)*

These helpers are the spine of the capstone and the same ones you'd want in any real project:

- **`load_slice` / `load_roi`** — read a course `.npz`, per-spectrum max-normalize.
- **`stratified_split`** — split **within each class** so both sides keep the rare class, and
  return **disjoint** index sets (no sample is in both train and val — that is *leakage*).
- **`honest_eval`** — the Lecture 10 report: **sensitivity** (caught / truly-positive),
  **specificity** (correctly-cleared / truly-negative), **AUROC**, and a **confusion matrix** —
  *not* bare accuracy, because every set here is imbalanced.


In [ ]:
# ==== read and run — the shared pipeline every track reuses ====
RAW_BASE = "https://raw.githubusercontent.com/<ORG>/<REPO>/main/data/slices/"

def resolve_slice(name):
    """Find a local course slice; fall back to the hosted copy on Colab."""
    for base in [Path("../../data/slices"), Path("data/slices"), Path.cwd() / "data" / "slices"]:
        p = base / name
        if p.exists():
            return p
    p = Path(name)                       # Colab: download next to the notebook
    if not p.exists():
        urllib.request.urlretrieve(RAW_BASE + name, p)
    return p

def load_slice(name):
    """Load a 6000-dim spectra slice; per-spectrum max-normalize to 0..1 (matches Lab 2)."""
    d = np.load(resolve_slice(name))
    X = d["X"].astype("float32")
    X = X / X.max(axis=1, keepdims=True)
    y = d["y"].astype("int64")
    return X, y

def load_roi(name):
    """Load the PeakOnly ROI slice (256-dim, already max-normalized); also return quality labels."""
    d = np.load(resolve_slice(name))
    return d["X"].astype("float32"), d["y"].astype("int64"), d["quality"].astype("int64")

def stratified_split(y, val_frac, seed=0):
    """Split indices within each class -> disjoint (train_idx, val_idx), >=1 per class per side."""
    y = np.asarray(y)
    rng = np.random.default_rng(seed)
    tr, va = [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0].copy()
        rng.shuffle(idx)
        n_val = max(1, int(round(val_frac * len(idx))))
        n_val = min(n_val, len(idx) - 1)     # always keep >=1 in train too
        va.extend(idx[:n_val].tolist())
        tr.extend(idx[n_val:].tolist())
    tr, va = np.array(tr), np.array(va)
    rng.shuffle(tr)
    rng.shuffle(va)
    return tr, va

def subset_stratified(X, y, max_n, seed=0):
    """Smoke helper: keep ~max_n samples, preserving class ratio (>=1 of each class)."""
    if max_n is None or max_n >= len(y):
        return X, y
    keep, _ = stratified_split(y, val_frac=1 - max_n / len(y), seed=seed)
    return X[keep], y[keep]

def spectra_tensor(X):
    return torch.tensor(X, dtype=torch.float32).unsqueeze(1)   # (N, 1, L) for Conv1d

def y_tensor(y):
    return torch.tensor(y, dtype=torch.float32).unsqueeze(1)   # (N, 1) for BCEWithLogits

class BaselineCNN(nn.Module):
    """The Lab 2 1D-CNN: two conv blocks + a two-layer head. Body is reusable for transfer."""
    def __init__(self, in_len=6000):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 8, 15, padding=7), nn.ReLU(), nn.MaxPool1d(4),    # L -> L/4
            nn.Conv1d(8, 16, 15, padding=7), nn.ReLU(), nn.MaxPool1d(4),   # L/4 -> L/16
            nn.Flatten(),
        )
        self.feat_dim = 16 * (in_len // 16)
        self.head = nn.Sequential(nn.Linear(self.feat_dim, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        return self.head(self.features(x))

class TransferModel(nn.Module):
    """A frozen (or shared) feature body + a fresh trainable head (Lecture 8 recipe)."""
    def __init__(self, body, head):
        super().__init__()
        self.body = body
        self.head = head
    def forward(self, x):
        return self.head(self.body(x))

def train_model(model, X, y, epochs, batch_size, lr, device,
                pos_weight=None, params=None, augment=None):
    """Reused mini-batch loop (Lecture 2 recipe). Optionally weight the rare class / augment."""
    model.to(device)
    X, y = X.to(device), y.to(device)
    pw = pos_weight.to(device) if pos_weight is not None else None
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt = torch.optim.Adam(params if params is not None else model.parameters(), lr=lr)
    n = X.shape[0]
    hist = []
    for _ in range(epochs):
        order = torch.randperm(n)
        tot = 0.0
        model.train()
        for s in range(0, n, batch_size):
            idx = order[s:s + batch_size]
            xb, yb = X[idx], y[idx]
            if augment is not None:
                xb = augment(xb)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item() * len(idx)
        hist.append(tot / n)
    return hist

def honest_eval(model, X, y, device, threshold=0.5):
    """The Lecture 10 report: sensitivity, specificity, AUROC + confusion matrix (not accuracy)."""
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X.to(device))).cpu().numpy().ravel()
    truth = np.asarray(y).ravel().astype(int)
    pred = (probs > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(truth, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else float("nan")     # recall of the positive class
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    auroc = roc_auc_score(truth, probs) if len(np.unique(truth)) == 2 else float("nan")
    return {"sensitivity": sens, "specificity": spec, "auroc": auroc,
            "confusion": np.array([[tn, fp], [fn, tp]]), "probs": probs, "truth": truth}

def print_report(name, rep):
    print(f"[{name}]  sensitivity={rep['sensitivity']:.3f}  "
          f"specificity={rep['specificity']:.3f}  AUROC={rep['auroc']:.3f}")
    cm = rep["confusion"]
    print(f"        confusion  [[TN {cm[0,0]}, FP {cm[0,1]}], [FN {cm[1,0]}, TP {cm[1,1]}]]")

# --- self-checks: do not edit ---
_pX = torch.randn(60, 4)
_py = np.array([0] * 30 + [1] * 30)
_probe = nn.Linear(4, 1)
_rep = honest_eval(_probe, _pX, _py, device=torch.device("cpu"))
for _k in ("sensitivity", "specificity", "auroc"):
    assert 0.0 <= _rep[_k] <= 1.0, f"{_k} must be a fraction in [0, 1]"
_tr, _va = stratified_split(_py, val_frac=0.25, seed=1)
assert len(set(_tr.tolist()) & set(_va.tolist())) == 0, "train/val must be disjoint — no leakage"
assert len(_tr) + len(_va) == len(_py), "split must cover every sample exactly once"
assert (_py[_va] == 1).sum() >= 1 and (_py[_va] == 0).sum() >= 1, "val must keep both classes"
print("Scaffolding OK — metrics in [0,1]; split is disjoint, complete, and keeps both classes.")


### Load the course data and build leakage-free splits *(read and run)*

We load all three slices, optionally shrink them for a smoke run, and build a stratified
80/20 split per track. The `assert`s prove each split is **disjoint and complete** — the
first thing a reviewer checks in a clinical model.


In [ ]:
# ==== read and run — load slices, subset (smoke), and split with no leakage ====
# Track A + C source: S. aureus / oxacillin (MRSA) — 6000-dim MALDI-TOF spectra
XA, yA = load_slice("driams_c_saureus_oxacillin.npz")
# Track C target: E. coli / ceftriaxone — a DIFFERENT organism/drug, same 6000-dim format
XC, yC = load_slice("driams_c_ecoli_ceftriaxone.npz")
# Track B: PeakOnly chromatographic ROIs — 256-dim windows, peak(1) vs noise(0) + quality
XB, yB, qualB = load_roi("peakonly_roi_qc.npz")

if SMOKE:                                   # shrink every set, preserving class ratio
    XA, yA = subset_stratified(XA, yA, MAX_N, seed=0)
    XC, yC = subset_stratified(XC, yC, MAX_N, seed=0)
    keepB, _ = stratified_split(yB, val_frac=1 - MAX_N / len(yB), seed=0)   # one index set
    XB, yB, qualB = XB[keepB], yB[keepB], qualB[keepB]   # keep spectra, labels, quality aligned

def make_split(y, seed=42):
    tr, va = stratified_split(y, val_frac=0.2, seed=seed)
    assert len(set(tr.tolist()) & set(va.tolist())) == 0, "leakage: a sample is in train AND val"
    assert len(tr) + len(va) == len(y), "split must cover every sample exactly once"
    return tr, va

trA, vaA = make_split(yA)
trB, vaB = make_split(yB)
trC, vaC = make_split(yC)

print(f"Track A  S.aureus/oxacillin : {len(yA)} spectra ({int(yA.sum())} R) -> train {len(trA)} / val {len(vaA)}")
print(f"Track B  PeakOnly ROIs      : {len(yB)} ROIs ({int(yB.sum())} peak) -> train {len(trB)} / val {len(vaB)}")
print(f"Track C  E.coli/ceftriaxone : {len(yC)} spectra ({int(yC.sum())} R) -> train {len(trC)} / val {len(vaC)}")


## Track A — Beat the baseline ✏️
*(only runs if `TRACK='A'` or `RUN_ALL_TRACKS=True`)*

The Lab 2 baseline (`BaselineCNN`) is provided. Your job: build an **improved** model and an
**augmentation**, then compare the two **on the exact same split** with `honest_eval`.

> **Note on imbalance.** The provided train cell up-weights the rare **resistant** class
> (`pos_weight`, the same trick as Track B) for *both* models. The S. aureus/oxacillin set is
> only ~5.6% resistant, so without weighting both models predict all-susceptible and
> sensitivity reads 0 — "beat the baseline" would then reduce to an AUROC-only ranking.

**Blank A1 — `ImprovedCNN`.** A deeper/wider 1D-CNN (three conv blocks, wider channels, an
adaptive pool so the head size is fixed). It must output one logit per spectrum, shape `(N, 1)`,
and — because it is a different network — have a **different trainable-parameter count** than
the baseline.

**Blank A2 — `augment`.** An **m/z-jitter** augmentation: shift each spectrum a few bins along
the mass axis and add a little intensity noise. It must **keep the shape** and **change the
values** (that's what makes it augmentation).


In [ ]:
if should_run("A"):
    # ---- Blank A1: an improved 1D-CNN (deeper/wider; adaptive pool -> fixed head size) ----
    ### BEGIN SOLUTION
    class ImprovedCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv1d(1, 16, 15, padding=7), nn.ReLU(), nn.MaxPool1d(4),
                nn.Conv1d(16, 32, 15, padding=7), nn.ReLU(), nn.MaxPool1d(4),
                nn.Conv1d(32, 32, 15, padding=7), nn.ReLU(),
                nn.AdaptiveMaxPool1d(8), nn.Flatten(),          # -> 32*8 = 256, fixed
            )
            self.head = nn.Sequential(nn.Linear(32 * 8, 64), nn.ReLU(), nn.Linear(64, 1))
        def forward(self, x):
            return self.head(self.features(x))
    ### END SOLUTION

    # ---- Blank A2: m/z-jitter augmentation (shift along the mass axis + a little noise) ----
    ### BEGIN SOLUTION
    def augment(x):
        shift = int(torch.randint(1, 4, (1,)).item())          # 1..3 bins of m/z jitter
        x = torch.roll(x, shifts=shift, dims=-1)
        x = x + 0.01 * torch.randn_like(x)
        return x
    ### END SOLUTION

    # --- self-checks: do not edit ---
    _base_p = sum(p.numel() for p in BaselineCNN().parameters())
    _imp_p = sum(p.numel() for p in ImprovedCNN().parameters())
    assert _imp_p != _base_p, "the improved model must differ from the baseline (different param count)"
    _out = ImprovedCNN()(torch.zeros(2, 1, 6000))
    assert _out.shape == (2, 1), f"improved model must output (N, 1) logits, got {tuple(_out.shape)}"
    _probe = torch.randn(2, 1, 6000)
    _aug = augment(_probe)
    assert _aug.shape == _probe.shape, "augmentation must preserve shape"
    assert not torch.allclose(_aug, _probe), "augmentation must actually change the values"
    print(f"Track A blanks OK — baseline params {_base_p:,} vs improved {_imp_p:,}; augment keeps shape, changes values.")


In [ ]:
if should_run("A"):
    XA_tr, yA_tr = spectra_tensor(XA[trA]), y_tensor(yA[trA])
    XA_va, yA_va = spectra_tensor(XA[vaA]), yA[vaA]

    # S. aureus/oxacillin is only ~5.6% resistant. Up-weight the rare resistant
    # class (same imbalance trick as Track B) so sensitivity is not pinned at 0 --
    # otherwise both models predict all-susceptible and "beat the baseline"
    # collapses to an AUROC-only ranking. Both models share this weighting, so the
    # comparison stays fair.
    n_pos_A = int((yA[trA] == 1).sum())
    n_neg_A = int((yA[trA] == 0).sum())
    pos_weight_A = torch.tensor([n_neg_A / n_pos_A])

    torch.manual_seed(0)
    baseline = BaselineCNN()
    train_model(baseline, XA_tr, yA_tr, EPOCHS, 32, 1e-3, device, pos_weight=pos_weight_A)
    base_rep = honest_eval(baseline, XA_va, yA_va, device)

    torch.manual_seed(0)
    improved = ImprovedCNN()
    train_model(improved, XA_tr, yA_tr, EPOCHS, 32, 1e-3, device,
                pos_weight=pos_weight_A, augment=augment)
    imp_rep = honest_eval(improved, XA_va, yA_va, device)

    print(f"Same split, class-weighted (pos_weight={pos_weight_A.item():.2f}), proper metrics (NOT accuracy):")
    print_report("baseline", base_rep)
    print_report("improved", imp_rep)


### Track A — honest error analysis *(read and run)*

The point isn't a single winning number — it's *where* each model fails. On imbalanced MRSA
data a model can look accurate while **missing resistant cases** (false negatives). Compare the
two confusion matrices and where the improved model changed the resistant calls.


In [ ]:
if should_run("A"):
    print("Honest error analysis — resistant (positive) class is the one that matters clinically.\n")
    for name, rep in [("baseline", base_rep), ("improved", imp_rep)]:
        cm = rep["confusion"]
        fn, tp = int(cm[1, 0]), int(cm[1, 1])
        fp = int(cm[0, 1])
        print(f"{name:>8}: caught {tp} resistant, MISSED {fn} (false negatives), "
              f"{fp} false alarms | AUROC {rep['auroc']:.3f}")
    print("\nAsk honestly: did 'improved' trade false alarms for fewer missed resistant cases,")
    print("or just move noise around? With one small site and few epochs, AUROC is the fair judge;")
    print("the confusion matrix says WHICH errors changed. Report both, never accuracy alone.")


## Track B — Peak QC ✏️
*(only runs if `TRACK='B'` or `RUN_ALL_TRACKS=True`)*

The PeakOnly ROIs are 256-point chromatographic windows labeled **real peak (1) vs. noise (0)** —
the Lecture 5 detection segment made real. Build a small classifier and handle the imbalance
(noise outnumbers peaks) honestly.

**Blank B1 — `ROIClassifier`.** A small 1D-CNN reading one 256-point ROI (`(N, 1, 256)`) and
outputting one logit `(N, 1)`.

**Blank B2 — `pos_weight`.** The class-imbalance handling for `BCEWithLogitsLoss`: up-weight the
rare positive (peak) class by the **negative-to-positive ratio** in the training set, as a
`torch.tensor([...])`. Because noise outnumbers peaks, this weight must be **> 1**.


In [ ]:
if should_run("B"):
    n_pos_train = int((yB[trB] == 1).sum())     # real peaks in the training split
    n_neg_train = int((yB[trB] == 0).sum())     # noise ROIs in the training split

    # ---- Blank B1: a small 1D-CNN for a 256-point ROI -> one logit ----
    ### BEGIN SOLUTION
    class ROIClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(1, 8, 7, padding=3), nn.ReLU(), nn.MaxPool1d(4),    # 256 -> 64
                nn.Conv1d(8, 16, 7, padding=3), nn.ReLU(), nn.MaxPool1d(4),   # 64 -> 16
                nn.Flatten(), nn.Linear(16 * 16, 32), nn.ReLU(), nn.Linear(32, 1),
            )
        def forward(self, x):
            return self.net(x)
    ### END SOLUTION

    # ---- Blank B2: up-weight the rare peak class by the neg:pos ratio ----
    ### BEGIN SOLUTION
    pos_weight = torch.tensor([n_neg_train / n_pos_train])
    ### END SOLUTION

    # --- self-checks: do not edit ---
    _out = ROIClassifier()(torch.zeros(2, 1, 256))
    assert _out.shape == (2, 1), f"ROI classifier must output (N, 1) logits, got {tuple(_out.shape)}"
    assert pos_weight.numel() == 1 and torch.isfinite(pos_weight).all(), "pos_weight must be one finite number"
    assert pos_weight.item() > 1.0, "noise outnumbers peaks — the rare class must be up-weighted (>1)"
    print(f"Track B blanks OK — logits (N,1); pos_weight={pos_weight.item():.2f} "
          f"(noise {n_neg_train} : peak {n_pos_train}).")


In [ ]:
if should_run("B"):
    XB_tr, yB_tr = spectra_tensor(XB[trB]), y_tensor(yB[trB])
    XB_va, yB_va = spectra_tensor(XB[vaB]), yB[vaB]

    torch.manual_seed(0)
    roi_model = ROIClassifier()
    train_model(roi_model, XB_tr, yB_tr, EPOCHS, 64, 1e-3, device, pos_weight=pos_weight)
    roi_rep = honest_eval(roi_model, XB_va, yB_va, device)

    print("Peak-vs-noise QC — proper metrics (NOT accuracy):")
    print_report("ROI classifier", roi_rep)


### Track B — honest error analysis *(read and run)*

Not every peak is equally easy. The slice ships **quality sub-labels** (good / low-intensity /
lousy / noisy). A clinically honest QC report asks: *which kinds* of peaks does the model miss?
We break the validation errors down by quality label.


In [ ]:
if should_run("B"):
    QUAL_NAMES = {0: "noise", 1: "good", 2: "low-intensity", 3: "lousy", 4: "noisy"}
    probs = roi_rep["probs"]
    pred = (probs > 0.5).astype(int)
    truth = roi_rep["truth"]
    qual_va = qualB[vaB]

    miss_peak = (truth == 1) & (pred == 0)        # real peaks called noise (false negatives)
    print("Honest error analysis — WHICH real peaks got missed, by quality sub-label:\n")
    for q in sorted(set(qual_va[truth == 1].tolist())):
        total = int(((truth == 1) & (qual_va == q)).sum())
        missed = int((miss_peak & (qual_va == q)).sum())
        print(f"  {QUAL_NAMES.get(q, q):>14}: missed {missed} / {total} real peaks")
    print(f"\nFalse alarms (noise called peak): {int(((truth == 0) & (pred == 1)).sum())}")
    print("Expect the 'low-intensity' and 'lousy' peaks to be hardest — that's an honest QC finding,")
    print("and it tells you where MORE labeled data or a better feature would help.")


## Track C — New question, same spectra ✏️
*(only runs if `TRACK='C'` or `RUN_ALL_TRACKS=True`)*

Transfer learning, the Lecture 8 recipe: a CNN **body** trained on one task already knows how to
read spectra. We train that body on **S. aureus/oxacillin**, then **freeze** it and attach a
**new head** for a *different* task — **E. coli/ceftriaxone** (new organism, new drug) — training
only the fresh head (the body stays frozen) with a **smaller** fine-tune learning rate.

First we train the S. aureus body *(read and run)*:


In [ ]:
if should_run("C"):
    XA_tr_c, yA_tr_c = spectra_tensor(XA[trA]), y_tensor(yA[trA])
    torch.manual_seed(0)
    saureus_model = BaselineCNN()             # will train, then donate its frozen body
    train_model(saureus_model, XA_tr_c, yA_tr_c, EPOCHS, 32, 1e-3, device)
    print("S. aureus body trained — ready to re-head for the E. coli task.")


**Blank C1 — freeze the body, attach a new head.** Reuse `saureus_model.features` as the
body, set every body parameter's `requires_grad = False` (so it stops learning), and build a
**fresh head** whose final layer outputs **one** logit for the new binary task. Wrap them in
`TransferModel`.

**Blank C2 — a smaller fine-tune LR.** Fine-tuning a fresh head on top of frozen features wants a
**gentler** step than training from scratch — a value `> 0` and `<= 1e-2` (smaller than the `1e-3`
the body was trained with is a good default).


In [ ]:
if should_run("C"):
    # ---- Blank C1: freeze the body, attach a fresh 1-logit head, wrap in TransferModel ----
    ### BEGIN SOLUTION
    transfer_body = saureus_model.features
    for p in transfer_body.parameters():
        p.requires_grad = False
    transfer_head = nn.Sequential(
        nn.Linear(saureus_model.feat_dim, 32), nn.ReLU(), nn.Linear(32, 1),
    )
    transfer_model = TransferModel(transfer_body, transfer_head)
    ### END SOLUTION

    # ---- Blank C2: a smaller learning rate for fine-tuning the new head ----
    ### BEGIN SOLUTION
    TRANSFER_LR = 5e-4
    ### END SOLUTION

    # --- self-checks: do not edit ---
    n_body = sum(p.numel() for p in transfer_body.parameters())
    n_frozen = sum(p.numel() for p in transfer_body.parameters() if not p.requires_grad)
    assert n_frozen == n_body, "the WHOLE body must be frozen (requires_grad=False on every body param)"
    n_trainable = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
    n_head = sum(p.numel() for p in transfer_head.parameters())
    assert n_trainable == n_head, "only the new head should be trainable (body is frozen)"
    _out = transfer_model(torch.zeros(2, 1, 6000).to(device))
    assert _out.shape == (2, 1), f"the re-headed model must output (N, 1) logits, got {tuple(_out.shape)}"
    assert 0 < TRANSFER_LR <= 1e-2, "use a small, positive fine-tune LR (<= 1e-2)"
    print(f"Track C blanks OK — frozen body {n_frozen:,} params, trainable head {n_trainable:,} params, "
          f"fine-tune LR {TRANSFER_LR}.")


In [ ]:
if should_run("C"):
    XC_tr, yC_tr = spectra_tensor(XC[trC]), y_tensor(yC[trC])
    XC_va, yC_va = spectra_tensor(XC[vaC]), yC[vaC]

    head_params = [p for p in transfer_model.parameters() if p.requires_grad]
    train_model(transfer_model, XC_tr, yC_tr, EPOCHS, 32, TRANSFER_LR, device, params=head_params)
    transfer_rep = honest_eval(transfer_model, XC_va, yC_va, device)

    print("Re-headed on E. coli/ceftriaxone — proper metrics (NOT accuracy):")
    print_report("transfer", transfer_rep)


### Track C — honest error analysis *(read and run)*

Transfer is not magic: a body trained on S. aureus may or may not carry over to E. coli. The
honest question is whether the frozen features are **good enough** for the new label, and which
class the errors fall on.


In [ ]:
if should_run("C"):
    cm = transfer_rep["confusion"]
    fn, tp, fp = int(cm[1, 0]), int(cm[1, 1]), int(cm[0, 1])
    print("Honest error analysis — did the FROZEN S. aureus body transfer to E. coli?\n")
    print(f"  resistant caught {tp}, MISSED {fn} (false negatives), {fp} false alarms")
    print(f"  AUROC {transfer_rep['auroc']:.3f} (0.5 = the frozen features carry no signal for this task)")
    print("\nIf AUROC sits near 0.5, the S. aureus features didn't transfer — the honest next step is")
    print("to UNFREEZE and fine-tune the body, or train from scratch. If it clears 0.5, transfer bought")
    print("you a working head from far less data than training a full CNN on E. coli would need.")


## Rubric self-check ✅ *(fill in for your track)*

Score yourself honestly (this is the same rubric the instructor uses). Aim for a *yes* on all four.

| # | Criterion | Ask yourself | ✔ |
|---|---|---|---|
| 1 | **Data handling** | Did I load the right slice, normalize, and keep train/val **disjoint** (no leakage)? The `make_split` `assert`s prove disjointness. | |
| 2 | **Model choice justified** | Can I say *in one sentence* why my model/augmentation/transfer choice fits **this** signal (spectrum vs. ROI vs. new label)? | |
| 3 | **Proper split & metrics** | Did I report **sensitivity, specificity, and AUROC + confusion matrix** — not bare accuracy — on the **held-out** split (Lecture 10)? | |
| 4 | **Honest error analysis** | Did I look at **which** errors happen (missed positives vs. false alarms; which quality/class) and say what I'd do next? | |

> A capstone with a great accuracy number but leakage or accuracy-only reporting **fails** this
> rubric. A modest AUROC with a clean split and an honest error story **passes**. That is the
> whole Lecture 10 lesson.


## Project one-pager 📄 *(your take-home template — fill it for YOUR lab)*

Copy this into a doc and complete it for a **first deep-learning project in your own lab**. A
printable version is in `labs/handouts/lab05_project_onepager.pdf`.

---

**1 · Problem.** *What decision or measurement would the model make, and who uses the output?*
> _(one or two sentences — e.g. "flag likely-resistant isolates from the MALDI spectrum before culture")_

**2 · Data & representation.** *What is one example, exactly? Shape, units, how many, how labeled,
and how imbalanced?*
> _(e.g. "one 6000-dim TIC-normalized spectrum; ~700 samples; 5–25% positive; labeled by AST")_

**3 · Architecture.** *What model, and why does it fit this representation?*
> _(e.g. "1D-CNN — peaks are local patterns anywhere on the m/z axis, like Lab 2 / Track A")_

**4 · Validation plan.** *What split (no leakage!), which metrics, and what target?*
> _(e.g. "stratified 80/20 by patient; sensitivity/specificity/AUROC; ≥0.90 sensitivity at a
> fixed specificity, chosen on a validation set — never a single accuracy number")_

**5 · Why not a simpler model?** *What baseline (logistic regression, a threshold, always-guess-majority)
must deep learning beat to be worth it?*
> _(if a one-line rule already hits your target, you may not need deep learning — that is a valid,
> honest conclusion and the Lecture 10 mindset)_

---


## Discussion D5 — closing (10 min)

**Share your one-pager.** In pairs, walk each other through your five boxes. Then a few
volunteers give a **60-second** version to the room.

Three questions to pressure-test each plan:
- **Leakage:** could any information from the validation set sneak into training (same patient,
  same batch, normalization fit on everything)?
- **The honest baseline:** what does "always guess the majority class" already score on your
  metric — and does deep learning clearly beat it?
- **The clinical cost:** is a missed positive (false negative) or a false alarm worse in your
  workflow, and does your **threshold** reflect that?

You now have a working pipeline, an honest evaluation habit, and a plan for your own data. That
is the whole point of DS301 — go apply it, and know when *not* to.
